# 🍽️ Weekly Themed Meal Planner MVP

A meal planning assistant that generates 7 themed dinners with consolidated shopping lists.

## Features
- 7 unique themed dinners (Monday-Sunday)
- 5 ingredients per recipe (excluding basics)
- Dietary preference support (vegan, vegetarian, keto, omnivore)
- Consolidated shopping list
- Email delivery

##Step 1 Notes:
- Normal run: generated a 7-day omnivore plan in 19.03 seconds, validation passed (screenshot saved)
- Missing .env: OPENAI_API_KEY becomes None, and the next cell crashes with a TypeError that does not say the key is missing (tested fully in Step 5)
- Hardcoded secrets: EMAIL_SENDER and EMAIL_PASSWORD in the config cell
- Email with placeholder settings: "Failed to send email: Connection unexpectedly closed". The message does not say the sender email/password are not set. The button also generates a NEW plan first (extra API call, different from the plan on screen).
- Unused package: langchain-anthropic

## Step 2: Code analysis

| # | Where | Category | Problem | Priority |
|---|---|---|---|---|
| 1 | Configuration cell | Hardcoded values | `EMAIL_SENDER` and `EMAIL_PASSWORD` are typed in the notebook; pushing to GitHub would leak them | High |
| 2 | Configuration cell | Missing error handling | If the API key is missing, the next cell crashes with a `TypeError` that never says the key is missing | High |
| 3 | `generate()`, `recipe_chain.invoke(...)` | Missing error handling | The API call is outside the `try`, so a wrong key or no internet crashes the app | High |
| 4 | `send_email_ui()` | Repeated code + bug | Email button generates a NEW plan, so the emailed plan is different from the one on screen (and costs an extra API call) | High |
| 5 | `generate()`, `except` blocks | Silent failure | Error messages say *what* went wrong but not *where*; `except Exception` treats all errors the same | High |
| 6 | `generate()` | Monolithic function | One method does 5 jobs: timer, API call, strip fences, parse JSON + validate, build result | Medium |
| 7 | `generate()`, code-fence stripping | Fragile logic | `split("```")` breaks if the model writes text before the JSON or forgets the closing fence | Medium |
| 8 | `generate()`, return dicts | Inconsistent output | Success dict has `generation_time`, failure dicts don't | Medium |
| 9 | `generate_plan_streaming()` and `send_email_ui()` | Repeated code | Same "if empty, use default" logic copy-pasted; `.strip()` crashes if the input is `None` | Medium |
| 10 | `format_meal_plan_email()` and `format_single_recipe()` | Repeated code | Same line turns ingredients into text in both | Low |
| 11 | `run_basic_validation()` vs Pydantic models | Duplicated logic + hardcoded values | Same rules (7 recipes, 5 ingredients, unique themes) checked twice; numbers 5 and 7 typed by hand instead of using constants | Low |
| 12 | `__init__`, prompts cell, config, install | Dead code | `shopping_parser`, `SHOPPING_LIST_PROMPT`, `EXCLUDED_BASICS`, `DAYS`, `DietaryPreference` and `langchain-anthropic` are never used | Low |
| 13 | `send_meal_plan_email()` and test cases cell | Mixed concerns | Email function builds the message AND sends it; test cell mixes test data, loop and printing, and makes 4 real API calls | Low |

### Top priorities
1. **Remove hardcoded secrets** (#1): move email settings to `.env` so nothing private goes to GitHub.
2. **Protect the API call** (#3, #5): put it inside `try` and give clear errors that say what, where, and how to fix it.
3. **Fix the email bug** (#4): email the plan that is already on screen instead of generating a new one.
4. **Split the monolithic `generate()`** (#6): break it into small functions with one job each, so each part can be tested on its own.

## Step 3: Helper functions
- `format_ingredients()`: replaces the copy-pasted ingredient line in the email and the UI
- `with_default()`: replaces the "if empty, use default" logic in 3 places; also fixes the crash when input is `None`
- `extract_json()`: replaces the fragile `split("```")` logic; works even with text before the JSON or a missing closing fence
- `format_error()`: one format for all error messages (used in Step 5)
- All helpers tested with fake data in a test cell (no API calls)
- Re-ran the app: same result as baseline (7 recipes, all checks passed)

## Step 4: Modularize

| Function | Describe it in one sentence | How many "and"s? |
|---|---|---|
| `generate()` | Starts a timer, builds the inputs **and** calls the API **and** strips fences **and** parses JSON **and** validates **and** builds the result | 5 |
| `send_meal_plan_email()` | Builds the email message **and** logs in to Gmail **and** sends it | 2 |
| `generate_plan_streaming()` | Generates a plan **and** formats it **and** streams it **and** runs validation | 3 |
| `send_email_ui()` | Checks the email address **and** generates a NEW plan **and** sends it | 2 |

### Changes
- `generate()` split into `build_inputs()`, `call_llm()`, `parse_meal_plan()` and `make_result()`; it now only calls them in order
- `make_result()` always returns the same 4 keys (fixes inconsistent output)
- `send_meal_plan_email()` split into `build_email_message()` and `send_email_message()`; Gmail server and port moved to constants
- `generate_plan_streaming()` uses new small helpers `format_plan_header()` and `format_status_line()`
- **Bug fixed:** the Email button now sends the plan shown on screen (stored in `gr.State`) instead of generating a new one
- `send_email_ui()` now says "Generate a meal plan first" if no plan exists yet
- Removed unused `shopping_parser`
- All new parts tested with fake data, no API calls
- Tested: Email button now responds in 1–2 seconds instead of ~19 seconds (no new plan generated)

## Step 5: Error handling

| Break this | How | What does the original show? | What does mine show? |
|---|---|---|---|
| OpenAI key | Delete it | `TypeError` in the config cell, does not say the key is missing | `OSError: [config] Missing setting: ... | Suggestion: Add ... to your .env file` (tested with `require_env()`, because my key is also set outside the notebook) |
| OpenAI key | Fake key | API call outside `try`; app crashes with a generic Gradio "Error" | `[MealPlanGenerator.call_llm] AuthenticationError: Incorrect API key provided | Suggestion: Check OPENAI_API_KEY in .env` |
| Network | Wi-Fi off, click Generate | Same crash: generic "Error" | `[MealPlanGenerator.call_llm] APIConnectionError | Suggestion: Check your internet connection` |
| JSON from the model | `'{"recipes": [}'` | Could not be tested without a real API call | `[parse_meal_plan] JSONDecodeError: Expecting value at line 1, column 14 | Suggestion: Click Generate again` |
| Meal-plan schema | 6 recipes | Long Pydantic message, no location | `[parse_meal_plan (MealPlan check)] ValidationError: recipes: must have exactly 7 recipes, got 6` |
| Meal-plan schema | 4 ingredients | Long Pydantic message, no location | `[parse_meal_plan (MealPlan check)] ValidationError: recipes.0.ingredients: must have exactly 5 ingredients, got 4` |
| Email | Settings missing / wrong password | "Connection unexpectedly closed" | `[send_meal_plan_email] Email settings missing | Suggestion: add EMAIL_SENDER and EMAIL_PASSWORD to .env` |
| Empty textbox | Empty Themes / `None` | `None` crashed with `AttributeError` | Both use the default: `themes = random`, `preferences = none` |

### Changes
- Email secrets moved from the notebook to `.env`; `.env` added to `.gitignore`
- `require_env()` stops right away with a clear message if the API key is missing
- The API call is now inside `try`, with a separate message for wrong key, rate limit, timeout, no internet and other API errors
- JSON errors show the exact line and column; Pydantic errors show exactly which field failed
- No bare `except:` and no `except Exception` left, so unknown errors are not hidden
- Errors that should stop the program (missing API key) use `raise`; errors the user can fix in the app are returned as clear messages in the UI
- All break tests except the fake-key test run with a fake model answer (no API calls)

- Missing API key: my OPENAI_API_KEY is also set outside the notebook, so renaming .env did not remove it. Tested `require_env()` with a missing test setting instead: it stops with "OSError: [config] Missing setting: TEST_MISSING_KEY | Suggestion: Add TEST_MISSING_KEY to your .env file" (EnvironmentError is the same as OSError in Python 3)

In [1]:
%pip install -q langchain langchain-openai langchain-anthropic pydantic gradio python-dotenv

Note: you may need to restart the kernel to use updated packages.


- Missing .env: OPENAI_API_KEY becomes None, and the next cell crashes with a TypeError that does not say the key is missing (tested fully in Step 5)

## 2. Configuration

Create a file called `.env` in the same folder as this notebook with this line:

```
OPENAI_API_KEY=sk-...
```

**Edit the values below before running the notebook.**

In [2]:
# ============================================================
# CONFIGURATION
# Secrets come from the .env file, never from this notebook.
# ============================================================
import os
from dotenv import load_dotenv

load_dotenv(".env")


def require_env(name: str, suggestion: str) -> str:
    """Return a required setting from .env, or stop with a clear message."""
    value = os.getenv(name)
    if not value:
        raise EnvironmentError(f"[config] Missing setting: {name} | Suggestion: {suggestion}")
    return value


# API key (required - the app cannot work without it)
OPENAI_API_KEY = require_env("OPENAI_API_KEY", "Add OPENAI_API_KEY=sk-... to your .env file")

# Email settings (optional - checked only when you click Email Plan)
EMAIL_SENDER = os.getenv("EMAIL_SENDER")
EMAIL_PASSWORD = os.getenv("EMAIL_PASSWORD")

# Model settings
OPENAI_MODEL = "gpt-4o-mini"
TEMPERATURE = 0.7

# Meal generation settings
INGREDIENTS_PER_RECIPE = 5
EXCLUDED_BASICS = ["salt", "pepper", "oil", "garlic", "water"]
DAYS = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]

In [3]:
# Set environment variables
os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY

## Helper functions.

In [4]:
# ============================================================
# HELPER FUNCTIONS
# Small functions with one job each, used in many places.
# ============================================================

# Default values for empty textboxes (one place instead of three)
DEFAULT_THEMES = "random"
DEFAULT_PREFERENCES = "none"


def format_ingredients(ingredients: list) -> str:
    """Turn a list of ingredients into one line, e.g. '200g pasta, 1 cup cheese'."""
    return ", ".join(f"{ing['quantity']} {ing['name']}" for ing in ingredients)


def with_default(value, default: str) -> str:
    """Return the cleaned value, or the default if it is empty or None."""
    if value is None or not str(value).strip():
        return default
    return str(value).strip()


def extract_json(text: str) -> str:
    """Return only the JSON part of the model's answer (from the first { to the last })."""
    start = text.find("{")
    end = text.rfind("}")
    if start == -1 or end == -1:
        return text.strip()
    return text[start:end + 1]


def format_error(where: str, error: Exception, suggestion: str, detail: str = None) -> str:
    """Build one clear error message: where, what type, what happened, how to fix it."""
    return f"[{where}] {type(error).__name__}: {detail or error} | Suggestion: {suggestion}"

In [5]:
# ============================================================
# TEST HELPERS with fake data (no OpenAI call, no cost)
# ============================================================

# format_ingredients
fake_ingredients = [
    {"name": "pasta", "quantity": "200g"},
    {"name": "cheese", "quantity": "1 cup"},
]
assert format_ingredients(fake_ingredients) == "200g pasta, 1 cup cheese"
print("✓ format_ingredients:", format_ingredients(fake_ingredients))

# with_default
assert with_default("", DEFAULT_THEMES) == "random"
assert with_default("   ", DEFAULT_THEMES) == "random"
assert with_default(None, DEFAULT_THEMES) == "random"
assert with_default(" Italian ", DEFAULT_THEMES) == "Italian"
print("✓ with_default: empty, spaces, None and normal text all work")

# extract_json
assert extract_json('```json\n{"a": 1}\n```') == '{"a": 1}'
assert extract_json('Here is your plan: {"a": 1}') == '{"a": 1}'
assert extract_json('```json\n{"a": 1}') == '{"a": 1}'   # missing closing fence
print("✓ extract_json: fences, text before JSON and missing fence all work")

# format_error
msg = format_error("test_cell", ValueError("bad value"), "Check your input")
print("✓ format_error:", msg)

print("\nAll helper tests passed!")

✓ format_ingredients: 200g pasta, 1 cup cheese
✓ with_default: empty, spaces, None and normal text all work
✓ extract_json: fences, text before JSON and missing fence all work
✓ format_error: [test_cell] ValueError: bad value | Suggestion: Check your input

All helper tests passed!


## 3. Pydantic Models for Validation

In [6]:
from pydantic import BaseModel, Field, field_validator
from typing import List, Optional
from enum import Enum


class DietaryPreference(str, Enum):
    OMNIVORE = "omnivore"
    VEGETARIAN = "vegetarian"
    VEGAN = "vegan"
    KETO = "keto"


class Ingredient(BaseModel):
    """Single ingredient with quantity."""
    name: str = Field(..., description="Name of the ingredient")
    quantity: str = Field(..., description="Quantity with unit (e.g., '2 cups', '500g')")


class Recipe(BaseModel):
    """A single themed dinner recipe."""
    day: str = Field(..., description="Day of the week")
    theme: str = Field(..., description="Cuisine theme (e.g., 'Italian', 'Mexican')")
    dish_name: str = Field(..., description="Name of the dish")
    ingredients: List[Ingredient] = Field(..., description="List of 5 ingredients")
    instructions: str = Field(..., description="Brief cooking instructions")

    @field_validator('ingredients')
    @classmethod
    def validate_ingredient_count(cls, v):
        if len(v) != INGREDIENTS_PER_RECIPE:
            raise ValueError(f"Recipe must have exactly {INGREDIENTS_PER_RECIPE} ingredients, got {len(v)}")
        return v


class ShoppingItem(BaseModel):
    """Consolidated shopping list item."""
    name: str = Field(..., description="Ingredient name")
    total_quantity: str = Field(..., description="Total quantity needed")
    used_in: List[str] = Field(..., description="List of dishes using this ingredient")


class MealPlan(BaseModel):
    """Complete weekly meal plan."""
    dietary_preference: str = Field(..., description="Selected dietary preference")
    recipes: List[Recipe] = Field(..., description="List of 7 recipes for the week")
    shopping_list: List[ShoppingItem] = Field(..., description="Consolidated shopping list")

    @field_validator('recipes')
    @classmethod
    def validate_recipe_count(cls, v):
        if len(v) != 7:
            raise ValueError(f"Meal plan must have exactly 7 recipes, got {len(v)}")
        return v

    @field_validator('recipes')
    @classmethod
    def validate_unique_themes(cls, v):
        themes = [r.theme.lower() for r in v]
        if len(themes) != len(set(themes)):
            raise ValueError("All 7 themes must be unique")
        return v

## 4. LangChain Setup & Prompts

In [7]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import JsonOutputParser


def get_llm():
    """Get the configured LLM."""
    return ChatOpenAI(model=OPENAI_MODEL, temperature=TEMPERATURE)


# Main meal plan generation prompt
MEAL_PLAN_PROMPT = ChatPromptTemplate.from_messages([
    ("system", """You are a professional meal planner. Generate a weekly meal plan with EXACTLY 7 themed dinners.

STRICT RULES - FOLLOW EXACTLY:
1. Each recipe MUST have EXACTLY 5 main ingredients (do NOT count salt, pepper, oil, garlic, or water)
2. ALL 7 THEMES MUST BE COMPLETELY DIFFERENT - no duplicates allowed! Use: Italian, Mexican, Japanese, Indian, Thai, Greek, American, Chinese, French, Korean, Mediterranean, Middle Eastern, etc.
3. All ingredients MUST be real, commonly available items
4. Follow the dietary preference strictly
5. Provide realistic quantities for 2 servings

Dietary Guidelines:
- Omnivore: Any ingredients allowed
- Vegetarian: No meat or fish, eggs and dairy OK
- Vegan: No animal products at all
- Keto: Low carb, high fat, no grains/sugar/starchy vegetables

Output valid JSON matching this exact structure:
{format_instructions}"""),
    ("human", """Create a weekly meal plan:
- Diet: {dietary_preference}
- Theme requests: {theme_requests}
- Preferences: {additional_preferences}

IMPORTANT: Each day MUST have a DIFFERENT cuisine theme. Generate 7 unique themed dinners for Monday-Sunday.""")
])


# Shopping list consolidation prompt
SHOPPING_LIST_PROMPT = ChatPromptTemplate.from_messages([
    ("system", """You are a shopping list organizer. Consolidate ingredients from multiple recipes into a single shopping list.

RULES:
1. Combine same ingredients and sum their quantities
2. List which dishes use each ingredient
3. Use standard units (cups, tablespoons, grams, pieces)
4. Round up quantities for convenience

Output valid JSON matching this structure:
{format_instructions}"""),
    ("human", """Consolidate the shopping list from these recipes:
{recipes_json}""")
])

## 5. Meal Generation Logic

In [8]:
import json
import time
import openai
from pydantic import ValidationError


def make_result(success: bool, meal_plan=None, error=None, generation_time=None) -> dict:
    """Build a result dict that always has the same 4 keys."""
    return {
        "success": success,
        "meal_plan": meal_plan,
        "generation_time": generation_time,
        "error": error,
    }


def parse_meal_plan(raw_text: str) -> MealPlan:
    """Turn the model's text answer into a validated MealPlan."""
    data = json.loads(extract_json(raw_text))
    return MealPlan(**data)


def describe_validation_error(e: ValidationError) -> str:
    """Turn a Pydantic error into a short list of the fields that failed."""
    return "; ".join(
        f"{'.'.join(str(part) for part in err['loc'])}: {err['msg']}" for err in e.errors()
    )


class MealPlanGenerator:
    """Generates weekly themed meal plans."""

    def __init__(self):
        self.llm = get_llm()
        self.recipe_parser = JsonOutputParser(pydantic_object=MealPlan)

    def build_inputs(self, dietary_preference, theme_requests, additional_preferences) -> dict:
        """Build the input dictionary for the meal plan prompt."""
        return {
            "dietary_preference": dietary_preference,
            "theme_requests": with_default(theme_requests, DEFAULT_THEMES),
            "additional_preferences": with_default(additional_preferences, DEFAULT_PREFERENCES),
            "format_instructions": self.recipe_parser.get_format_instructions(),
        }

    def call_llm(self, inputs: dict) -> str:
        """Send the prompt to the model and return its text answer."""
        chain = MEAL_PLAN_PROMPT | self.llm
        return chain.invoke(inputs).content

    def generate(
        self,
        dietary_preference: str,
        theme_requests: str = DEFAULT_THEMES,
        additional_preferences: str = DEFAULT_PREFERENCES,
    ) -> dict:
        """Generate a meal plan; every failure becomes a clear error message."""
        start_time = time.time()

        # Step 1: call the API (now protected!)
        try:
            inputs = self.build_inputs(dietary_preference, theme_requests, additional_preferences)
            raw_text = self.call_llm(inputs)
        except openai.AuthenticationError as e:
            return make_result(False, error=format_error(
                "MealPlanGenerator.call_llm", e, "Your OpenAI API key is wrong. Check OPENAI_API_KEY in .env"))
        except openai.RateLimitError as e:
            return make_result(False, error=format_error(
                "MealPlanGenerator.call_llm", e, "Too many requests or no credits left. Wait a minute or check your OpenAI billing"))
        except openai.APITimeoutError as e:
            return make_result(False, error=format_error(
                "MealPlanGenerator.call_llm", e, "OpenAI took too long to answer. Try again"))
        except openai.APIConnectionError as e:
            return make_result(False, error=format_error(
                "MealPlanGenerator.call_llm", e, "Cannot reach OpenAI. Check your internet connection"))
        except openai.APIError as e:
            return make_result(False, error=format_error(
                "MealPlanGenerator.call_llm", e, "OpenAI returned an error. Try again later"))

        # Step 2: parse and validate the answer
        try:
            meal_plan = parse_meal_plan(raw_text)
        except json.JSONDecodeError as e:
            return make_result(False, error=format_error(
                "parse_meal_plan", e, "The model returned broken JSON. Click Generate again",
                detail=f"{e.msg} at line {e.lineno}, column {e.colno}"))
        except ValidationError as e:
            return make_result(False, error=format_error(
                "parse_meal_plan (MealPlan check)", e, "The plan broke a rule. Click Generate again",
                detail=describe_validation_error(e)))

        elapsed = round(time.time() - start_time, 2)
        return make_result(True, meal_plan=meal_plan.model_dump(), generation_time=elapsed)


# Initialize generator
generator = MealPlanGenerator()

In [9]:
# ============================================================
# TEST parse_meal_plan and make_result with fake data (no API call)
# ============================================================

fake_plan = {
    "dietary_preference": "omnivore",
    "recipes": [
        {
            "day": day,
            "theme": f"Theme {i}",
            "dish_name": f"Dish {i}",
            "ingredients": [{"name": f"item {j}", "quantity": "1 cup"} for j in range(5)],
            "instructions": "Cook it.",
        }
        for i, day in enumerate(DAYS)
    ],
    "shopping_list": [{"name": "item 0", "total_quantity": "7 cups", "used_in": ["Dish 0"]}],
}

# The model sometimes writes text before the JSON - parse_meal_plan should still work
fake_answer = "Here is your plan:\n```json\n" + json.dumps(fake_plan) + "\n```"

plan = parse_meal_plan(fake_answer)
assert len(plan.recipes) == 7
print("✓ parse_meal_plan: 7 recipes parsed from a fake answer")

# Success and failure results have the same keys
ok = make_result(True, meal_plan={}, generation_time=1.0)
fail = make_result(False, error="something broke")
assert ok.keys() == fail.keys()
print("✓ make_result: success and failure have the same keys:", list(ok.keys()))

✓ parse_meal_plan: 7 recipes parsed from a fake answer
✓ make_result: success and failure have the same keys: ['success', 'meal_plan', 'generation_time', 'error']


## 6. Email Functionality

In [10]:
import smtplib
from email.mime.text import MIMEText
from email.mime.multipart import MIMEMultipart

SMTP_SERVER = "smtp.gmail.com"
SMTP_PORT = 587


def format_meal_plan_email(meal_plan: dict) -> str:
    """Format the meal plan as plain email text (shopping list first)."""
    lines = []
    lines.append("=" * 40)
    lines.append(f"WEEKLY MEAL PLAN ({meal_plan['dietary_preference'].upper()})")
    lines.append("=" * 40)
    lines.append("")

    lines.append("SHOPPING LIST")
    lines.append("-" * 20)
    for item in meal_plan['shopping_list']:
        lines.append(f"- {item['total_quantity']} {item['name']}")
    lines.append("")

    lines.append("=" * 40)
    lines.append("RECIPES")
    lines.append("=" * 40)
    for recipe in meal_plan['recipes']:
        lines.append(f"\n{recipe['day'].upper()} - {recipe['theme']}")
        lines.append(f"{recipe['dish_name']}")
        lines.append(f"Ingredients: {format_ingredients(recipe['ingredients'])}")
        lines.append(f"Instructions: {recipe['instructions']}")

    return "\n".join(lines)


def build_email_message(recipient_email: str, meal_plan: dict) -> MIMEMultipart:
    """Create the email message that contains the meal plan."""
    msg = MIMEMultipart()
    msg['From'] = EMAIL_SENDER
    msg['To'] = recipient_email
    msg['Subject'] = "Your Weekly Meal Plan"
    msg.attach(MIMEText(format_meal_plan_email(meal_plan), 'plain'))
    return msg


def send_email_message(msg: MIMEMultipart) -> None:
    """Log in to the email server and send a ready-made message."""
    with smtplib.SMTP(SMTP_SERVER, SMTP_PORT) as server:
        server.starttls()
        server.login(EMAIL_SENDER, EMAIL_PASSWORD)
        server.send_message(msg)


def send_meal_plan_email(recipient_email: str, meal_plan: dict) -> dict:
    """Send the meal plan by email and return a status dict."""
    if not EMAIL_SENDER or not EMAIL_PASSWORD:
        return {"success": False, "message": (
            "[send_meal_plan_email] Email settings missing | Suggestion: "
            "add EMAIL_SENDER and EMAIL_PASSWORD (a Gmail App Password) to .env")}

    try:
        msg = build_email_message(recipient_email, meal_plan)
        send_email_message(msg)
        return {"success": True, "message": f"Email sent to {recipient_email}"}
    except smtplib.SMTPAuthenticationError as e:
        message = format_error("send_email_message", e,
                               "Wrong email or password. Use a Gmail App Password, not your normal password")
    except smtplib.SMTPServerDisconnected as e:
        message = format_error("send_email_message", e,
                               "Gmail closed the connection, usually because of wrong login details. Check EMAIL_SENDER and EMAIL_PASSWORD")
    except (smtplib.SMTPException, OSError) as e:
        message = format_error("send_email_message", e,
                               "Could not reach the email server. Check your internet connection")
    return {"success": False, "message": message}

## 7. Validation

In [11]:
def run_basic_validation(meal_plan_result: dict) -> dict:
    """Run quick programmatic validation checks (not LLM-based)."""
    if not meal_plan_result.get("success"):
        return {"passed": False, "errors": [meal_plan_result.get("error")]}

    meal_plan = meal_plan_result["meal_plan"]
    errors = []

    # Check 1: 7 recipes
    if len(meal_plan["recipes"]) != 7:
        errors.append(f"Expected 7 recipes, got {len(meal_plan['recipes'])}")

    # Check 2: Unique themes
    themes = [r["theme"].lower() for r in meal_plan["recipes"]]
    if len(themes) != len(set(themes)):
        duplicates = [t for t in themes if themes.count(t) > 1]
        errors.append(f"Duplicate themes: {set(duplicates)}")

    # Check 3: 5 ingredients per recipe
    for recipe in meal_plan["recipes"]:
        if len(recipe["ingredients"]) != 5:
            errors.append(f"{recipe['day']}: {len(recipe['ingredients'])} ingredients (expected 5)")

    return {"passed": len(errors) == 0, "errors": errors}

## 8. Gradio Interface

In [12]:
import gradio as gr
import time as pytime


def format_shopping_list(meal_plan: dict) -> str:
    """Format the shopping list section."""
    lines = ["## Shopping List\n"]
    for item in meal_plan['shopping_list']:
        lines.append(f"- {item['total_quantity']} **{item['name']}**")
    return "\n".join(lines)


def format_single_recipe(recipe: dict) -> str:
    """Format one recipe compactly."""
    ingredients = format_ingredients(recipe['ingredients'])
    return f"**{recipe['day']}** | {recipe['theme']} | *{recipe['dish_name']}*\n{ingredients}\n"


def format_plan_header(meal_plan: dict) -> str:
    """Build the title and shopping list shown above the recipes."""
    header = f"## Weekly {meal_plan['dietary_preference'].title()} Meal Plan\n\n"
    header += format_shopping_list(meal_plan) + "\n\n---\n\n"
    header += "## Recipes\n\n"
    return header


def format_status_line(result: dict) -> str:
    """Build the footer with the generation time and validation result."""
    validation = run_basic_validation(result)
    status = "All checks passed" if validation["passed"] else f"Issues: {', '.join(validation['errors'])}"
    return f"---\n*Generated in {result['generation_time']}s | {status}*"


def generate_plan_streaming(dietary_pref, theme_requests, additional_prefs):
    """Stream a new meal plan to the screen (the plan is also kept for the email button)."""
    yield "Generating your meal plan...", None

    result = generator.generate(dietary_pref, theme_requests, additional_prefs)
    if not result["success"]:
        yield f"Error: {result['error']}", None
        return

    meal_plan = result["meal_plan"]
    output = format_plan_header(meal_plan)
    yield output, meal_plan
    pytime.sleep(0.1)

    for recipe in meal_plan['recipes']:
        output += format_single_recipe(recipe) + "\n"
        yield output, meal_plan
        pytime.sleep(0.05)

    output += format_status_line(result)
    yield output, meal_plan


def send_email_ui(email, current_plan):
    """Email the plan that is currently shown on screen."""
    if not email or "@" not in email:
        return "Enter a valid email address."
    if current_plan is None:
        return "Generate a meal plan first, then click Email Plan."
    return send_meal_plan_email(email, current_plan)["message"]


# Create Gradio interface
with gr.Blocks(title="Weekly Meal Planner") as app:
    gr.Markdown("# Weekly Meal Planner\nGenerate 7 themed dinners with a consolidated shopping list.")

    # Hidden "memory box" that keeps the plan shown on screen
    current_plan = gr.State(None)

    with gr.Row():
        with gr.Column(scale=1):
            dietary_dropdown = gr.Dropdown(
                choices=["omnivore", "vegetarian", "vegan", "keto"],
                value="omnivore",
                label="Diet"
            )
            theme_input = gr.Textbox(label="Themes (optional)", placeholder="Italian, Mexican...", lines=1)
            additional_input = gr.Textbox(label="Preferences (optional)", placeholder="no nuts, quick meals...", lines=1)
            generate_btn = gr.Button("Generate", variant="primary")

            gr.Markdown("---")
            email_input = gr.Textbox(label="Email", placeholder="you@example.com")
            send_btn = gr.Button("Email Plan")
            email_status = gr.Textbox(label="Status", interactive=False)

        with gr.Column(scale=2):
            output_display = gr.Markdown("Click **Generate** to start.")

    generate_btn.click(
        fn=generate_plan_streaming,
        inputs=[dietary_dropdown, theme_input, additional_input],
        outputs=[output_display, current_plan]
    )

    send_btn.click(
        fn=send_email_ui,
        inputs=[email_input, current_plan],
        outputs=[email_status]
    )

c:\Users\bbars\miniconda3_new\envs\bootcamp-env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [13]:
# ============================================================
# TEST email building and email button logic (no API, no email sent)
# ============================================================

fake_meal_plan = plan.model_dump()   # 'plan' comes from the Step 4b test cell

msg = build_email_message("test@example.com", fake_meal_plan)
assert msg["To"] == "test@example.com"
assert "SHOPPING LIST" in msg.get_payload()[0].get_payload()
print("✓ build_email_message: message built without sending anything")

print("✓ No email:      ", send_email_ui("", None))
print("✓ Bad email:     ", send_email_ui("not-an-email", None))
print("✓ No plan yet:   ", send_email_ui("test@example.com", None))

✓ build_email_message: message built without sending anything
✓ No email:       Enter a valid email address.
✓ Bad email:      Enter a valid email address.
✓ No plan yet:    Generate a meal plan first, then click Email Plan.


## 9. Launch Application

In [14]:
# Launch the Gradio app
app.launch(share=True)

* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://da65579fd62521f248.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


## 10. Run Test Cases

Runs every test case through the generator and the basic validation checks. This makes 4 real API calls.

In [26]:
# ============================================================
# ⚠️ REAL API TESTS - only run when RUN_REAL_API_TESTS = True
# (4 real API calls, costs money)
# ============================================================
RUN_REAL_API_TESTS = False

test_cases, error = load_test_cases("test_cases.json")

if error:
    print(error)
elif not RUN_REAL_API_TESTS:
    print(f"Loaded {len(test_cases)} test cases. Real API tests are OFF (set RUN_REAL_API_TESTS = True to run them).")
else:
    for case in test_cases:
        result = generator.generate(
            case["dietary_preference"],
            case.get("theme_requests"),
            case.get("additional_preferences"),
        )
        validation = run_basic_validation(result)
        print(case["dietary_preference"], "->", "PASS" if validation["passed"] else validation["errors"])

Loaded 4 test cases. Real API tests are OFF (set RUN_REAL_API_TESTS = True to run them).


## Step 5 tests

In [16]:
# ============================================================
# STEP 5 TESTS: break things on purpose (NO API calls, free)
# We swap call_llm for a fake that returns our own text.
# ============================================================
import copy


def run_with_fake_answer(fake_text: str) -> dict:
    """Run generate() with a fake model answer instead of calling OpenAI."""
    generator.call_llm = lambda inputs: fake_text
    try:
        return generator.generate("omnivore")
    finally:
        del generator.call_llm   # put the real call_llm back


# 1. Broken JSON from the model
r = run_with_fake_answer('{"recipes": [}')
print("1. Broken JSON:\n  ", r["error"], "\n")

# 2. Plan with only 6 recipes
six_recipes = copy.deepcopy(fake_plan)
six_recipes["recipes"] = six_recipes["recipes"][:6]
r = run_with_fake_answer(json.dumps(six_recipes))
print("2. Six recipes:\n  ", r["error"], "\n")

# 3. A recipe with only 4 ingredients
four_ingredients = copy.deepcopy(fake_plan)
four_ingredients["recipes"][0]["ingredients"] = four_ingredients["recipes"][0]["ingredients"][:4]
r = run_with_fake_answer(json.dumps(four_ingredients))
print("3. Four ingredients:\n  ", r["error"], "\n")

# 4. Empty textbox and None
inputs = generator.build_inputs("omnivore", None, "   ")
print("4. None / empty textbox:\n   themes =", inputs["theme_requests"],
      "| preferences =", inputs["additional_preferences"], "\n")

# 5. A good fake answer still works
r = run_with_fake_answer(json.dumps(fake_plan))
assert r["success"]
print("5. Valid plan still works:", r["success"], "\n")

# 6. Email settings missing (only runs if they are NOT set, so no real email is sent)
if not EMAIL_SENDER or not EMAIL_PASSWORD:
    print("6. Email settings missing:\n  ", send_meal_plan_email("test@example.com", fake_plan)["message"])
else:
    print("6. Skipped: email settings are set. Test a wrong password in the app instead.")

1. Broken JSON:
   [parse_meal_plan] JSONDecodeError: Expecting value at line 1, column 14 | Suggestion: The model returned broken JSON. Click Generate again 

2. Six recipes:
   [parse_meal_plan (MealPlan check)] ValidationError: recipes: Value error, Meal plan must have exactly 7 recipes, got 6 | Suggestion: The plan broke a rule. Click Generate again 

3. Four ingredients:
   [parse_meal_plan (MealPlan check)] ValidationError: recipes.0.ingredients: Value error, Recipe must have exactly 5 ingredients, got 4 | Suggestion: The plan broke a rule. Click Generate again 

4. None / empty textbox:
   themes = random | preferences = none 

5. Valid plan still works: True 

6. Email settings missing:
   [send_meal_plan_email] Email settings missing | Suggestion: add EMAIL_SENDER and EMAIL_PASSWORD (a Gmail App Password) to .env


In [17]:
# ============================================================
# ⚠️ REAL API TEST - sends ONE request to OpenAI with a FAKE key.
# It is rejected, so it costs nothing.
# ============================================================
from langchain_openai import ChatOpenAI

bad_generator = MealPlanGenerator()
bad_generator.llm = ChatOpenAI(model=OPENAI_MODEL, api_key="sk-fake-key-for-testing")

r = bad_generator.generate("omnivore")
print("Fake API key:\n  ", r["error"])

Fake API key:
   [MealPlanGenerator.call_llm] OpenAIAuthenticationError: Error code: 401 - {'error': {'message': 'Incorrect API key provided: sk-fake-***********ting. You can find your API key at https://platform.openai.com/account/api-keys.', 'type': 'invalid_request_error', 'code': 'invalid_api_key', 'param': None}, 'status': 401} | Suggestion: Your OpenAI API key is wrong. Check OPENAI_API_KEY in .env


In [18]:
# Simulate a missing setting without touching the real API key
os.environ.pop("TEST_MISSING_KEY", None)
require_env("TEST_MISSING_KEY", "Add TEST_MISSING_KEY to your .env file")

OSError: [config] Missing setting: TEST_MISSING_KEY | Suggestion: Add TEST_MISSING_KEY to your .env file

In [19]:
# ============================================================
# CREATE TEST FILES (run once)
# ============================================================
import json

# 1. A good file with the 4 test cases
TEST_CASES = [
    {"dietary_preference": "omnivore", "theme_requests": "random", "additional_preferences": "none"},
    {"dietary_preference": "vegetarian", "theme_requests": "Italian, Asian, Mexican", "additional_preferences": "no mushrooms"},
    {"dietary_preference": "vegan", "theme_requests": "random", "additional_preferences": "high protein"},
    {"dietary_preference": "keto", "theme_requests": "Mediterranean, American", "additional_preferences": "none"},
]
with open("test_cases.json", "w", encoding="utf-8") as f:
    json.dump(TEST_CASES, f, indent=2)

# 2. A broken file (JSON cut off in the middle)
with open("malformed.json", "w", encoding="utf-8") as f:
    f.write('[{"dietary_preference": "vegan",')

# 3. A file with bad data (unknown diet, missing diet)
with open("invalid_test_cases.json", "w", encoding="utf-8") as f:
    json.dump([{"dietary_preference": "carnivore"}, {"theme_requests": "Italian"}], f, indent=2)

print("✓ Created test_cases.json, malformed.json and invalid_test_cases.json")

✓ Created test_cases.json, malformed.json and invalid_test_cases.json


In [20]:
# ============================================================
# READ TEST CASES FROM A FILE (with clear errors)
# ============================================================

# Uses the DietaryPreference enum, which was dead code before
VALID_DIETS = [d.value for d in DietaryPreference]


def check_test_cases(data) -> list:
    """Return a list of problems in the test case data (empty list = all good)."""
    if not isinstance(data, list):
        return ["the file must contain a list of test cases"]
    problems = []
    for i, case in enumerate(data):
        diet = case.get("dietary_preference") if isinstance(case, dict) else None
        if diet not in VALID_DIETS:
            problems.append(f"case {i}: dietary_preference must be one of {VALID_DIETS}, got {diet!r}")
    return problems


def load_test_cases(path: str):
    """Read test cases from a JSON file; return (cases, error_message)."""
    try:
        with open(path, "r", encoding="utf-8") as f:
            data = json.load(f)
    except FileNotFoundError as e:
        return [], format_error("load_test_cases", e,
                                f"Check that '{path}' exists in the notebook folder")
    except json.JSONDecodeError as e:
        return [], format_error("load_test_cases", e, f"Fix the JSON in '{path}'",
                                detail=f"{e.msg} at line {e.lineno}, column {e.colno}")

    problems = check_test_cases(data)
    if problems:
        return [], format_error("load_test_cases", ValueError("invalid test data"),
                                f"Fix the test cases in '{path}'", detail="; ".join(problems))
    return data, None

In [21]:
# ============================================================
# TEST the file reader (no API calls)
# ============================================================

cases, error = load_test_cases("test_cases.json")
print("1. Good file:", len(cases), "test cases loaded | error:", error, "\n")

cases, error = load_test_cases("does_not_exist.json")
print("2. Missing file:\n  ", error, "\n")

cases, error = load_test_cases("malformed.json")
print("3. Broken file:\n  ", error, "\n")

cases, error = load_test_cases("invalid_test_cases.json")
print("4. Bad data:\n  ", error)

1. Good file: 4 test cases loaded | error: None 

2. Missing file:
   [load_test_cases] FileNotFoundError: [Errno 2] No such file or directory: 'does_not_exist.json' | Suggestion: Check that 'does_not_exist.json' exists in the notebook folder 

3. Broken file:
   [load_test_cases] JSONDecodeError: Expecting property name enclosed in double quotes at line 1, column 33 | Suggestion: Fix the JSON in 'malformed.json' 

4. Bad data:
   [load_test_cases] ValueError: case 0: dietary_preference must be one of ['omnivore', 'vegetarian', 'vegan', 'keto'], got 'carnivore'; case 1: dietary_preference must be one of ['omnivore', 'vegetarian', 'vegan', 'keto'], got None | Suggestion: Fix the test cases in 'invalid_test_cases.json'
